# 01 — Burn-in, snapshot e dataset diagnostico di transizioni

Questo notebook costruisce il primo contratto dati riproducibile del teacher Hay:

$$ (S_t, U_{[t,t+1 ms]}) \rightarrow (S_{t+1}, \text{microtraccia}, \text{eventi}) $$

Non addestra HayFlow e non introduce latenti, riduzione morfologica o generazione massiva. Il notebook fallisce esplicitamente se il teacher non coincide con l'audit o se una transizione non è riproducibile.

## 1. Checkout riproducibile

La prima cella eseguibile clona la repository di progetto quando il notebook gira su Kaggle/Colab. `HAYFLOW_ELM_REF` permette di selezionare un commit o branch preciso.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

ELM_REPOSITORY = "https://github.com/Zagred47/giada.git"
ELM_REF = os.environ.get("HAYFLOW_ELM_REF", "main")
if Path("/kaggle/working").is_dir():
    NOTEBOOK_ROOT = Path("/kaggle/working")
elif Path("/content").is_dir():
    NOTEBOOK_ROOT = Path("/content")
else:
    NOTEBOOK_ROOT = Path.cwd().resolve()
WORKSPACE = NOTEBOOK_ROOT / "hayflow_workspace"
WORKSPACE.mkdir(parents=True, exist_ok=True)

def run(command, cwd=None):
    print("+", " ".join(map(str, command)))
    subprocess.run(list(map(str, command)), cwd=cwd, check=True)

elm_override = os.environ.get("HAYFLOW_ELM_REPO")
mounted = [Path(elm_override).expanduser()] if elm_override else []
mounted.extend([Path.cwd(), *Path.cwd().parents])
mounted_elm = next((p.resolve() for p in mounted if (p / "src" / "hayflow_teacher").is_dir()), None)
if mounted_elm is not None:
    ELM_REPO = mounted_elm
else:
    ELM_REPO = WORKSPACE / "elmneuron"
    if not (ELM_REPO / ".git").is_dir():
        run(["git", "clone", ELM_REPOSITORY, ELM_REPO])
    run(["git", "fetch", "origin", ELM_REF], cwd=ELM_REPO)
    run(["git", "checkout", "--detach", "FETCH_HEAD"], cwd=ELM_REPO)
os.environ["HAYFLOW_ELM_REPO"] = str(ELM_REPO)
print("Owned repository:", ELM_REPO)

## 2. Teacher e dipendenze

Il teacher resta un checkout separato e bloccato al commit già approvato dall'audit. Compiliamo i MOD originali senza modificarli.

In [ ]:
import shutil

TEACHER_REPOSITORY = "https://github.com/SelfishGene/neuron_as_deep_net.git"
TEACHER_COMMIT = "074c4666300a8ad246601dab179a97a6942f0f29"
teacher_override = os.environ.get("HAYFLOW_TEACHER_REPO")
sibling = ELM_REPO.parent / "neuron_as_deep_net"
TEACHER_REPO = Path(teacher_override).expanduser().resolve() if teacher_override else sibling
if not (TEACHER_REPO / ".git").is_dir():
    run(["git", "clone", TEACHER_REPOSITORY, TEACHER_REPO])
run(["git", "checkout", "--detach", TEACHER_COMMIT], cwd=TEACHER_REPO)
assert subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=TEACHER_REPO, text=True).strip() == TEACHER_COMMIT
os.environ["HAYFLOW_TEACHER_REPO"] = str(TEACHER_REPO)

run([sys.executable, "-m", "pip", "install", "--quiet", "neuron==8.2.7", "numpy", "pandas", "matplotlib", "h5py", "pyarrow", "pyyaml"])
SIMULATION_DIR = TEACHER_REPO / "L5PC_NEURON_simulation"
compiled = list(SIMULATION_DIR.rglob("libnrnmech.so"))
if not compiled:
    nrnivmodl = shutil.which("nrnivmodl") or str(Path(sys.executable).parent / "nrnivmodl")
    run([nrnivmodl, "mods"], cwd=SIMULATION_DIR)
assert list(SIMULATION_DIR.rglob("libnrnmech.so")), "MOD compilation failed"
print("Teacher ready:", TEACHER_REPO)

## 3. Configurazione e istanziazione canonica

Questa fase ricostruisce lo stesso teacher dell'audit, verifica commit e hash, crea il manifest a 642 segmenti e prepara lo schema stabile delle variabili.

In [ ]:
import json
import yaml
from IPython.display import display

sys.path.insert(0, str(ELM_REPO))
from src.hayflow_data import BurnInCriteria
from src.hayflow_teacher import DiagnosticDatasetSession, expected_audit_hashes

CONFIG_PATH = ELM_REPO / "configs" / "hayflow" / "transition_dataset_diagnostic.yml"
config = yaml.safe_load(CONFIG_PATH.read_text(encoding="utf-8"))
OUTPUT_DIR = NOTEBOOK_ROOT / "hayflow_transition_dataset_diagnostic"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
shutil.copy2(CONFIG_PATH, OUTPUT_DIR / "experiment_config.yml")
assert config["teacher"]["expected_source_sha256"] == expected_audit_hashes()
session = DiagnosticDatasetSession(
    ELM_REPO,
    TEACHER_REPO,
    output_dir=OUTPUT_DIR,
    seed=config["runtime"]["seed"],
    expected_teacher_hashes=config["teacher"]["expected_source_sha256"],
)
teacher_report = session.prepare_teacher()
display(teacher_report)
assert teacher_report["segment_count"] == 642
assert teacher_report["teacher_hashes_match_audit"]

## 4. Burn-in misurato

Non imponiamo una durata. Ogni millisecondo misuriamo la massima variazione di voltaggio, calcio e STATE lenti. Lo snapshot viene accettato solo dopo che tutti i criteri restano veri per `consecutive_ms`. Il limite massimo è un guardrail: raggiungerlo è un errore, non una convergenza.

In [ ]:
burnin_config = dict(config["burnin"])
burnin_config["slow_mechanisms"] = tuple(burnin_config["slow_mechanisms"])
burnin_report = session.run_burn_in(BurnInCriteria(**burnin_config))
display({key: value for key, value in burnin_report.items() if key != "metrics"})
assert burnin_report["converged"]

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

burnin_metrics = pd.DataFrame(burnin_report["metrics"])
axes = burnin_metrics.plot(
    x="time_ms",
    y=["max_abs_voltage_delta_mv", "max_relative_calcium_delta", "max_abs_slow_state_delta"],
    logy=True, subplots=True, figsize=(12, 8), grid=True, title="Burn-in convergence metrics"
)
plt.tight_layout()
plt.savefig(session.figures_dir / "burnin_convergence.png", dpi=160)
plt.show()

## 5. Smoke test e calibrazione della corrente somatica

Prima di generare dati verifichiamo che la corrente dichiarata venga realmente erogata dall'`IClamp` e modifichi il voltaggio somatico. Poi scegliamo automaticamente la minima ampiezza testata che produca sia uno spike AIS sia uno spike somatico con il protocollo a due impulsi.

In [ ]:
current_smoke = session.run_somatic_current_smoke_test(**config["somatic_current"]["smoke_test"])
display(current_smoke)
assert current_smoke["valid"]

somatic_calibration = session.calibrate_somatic_spike_current(
    candidate_amplitudes_na=config["somatic_current"]["calibration"]["candidate_amplitudes_na"]
)
display(somatic_calibration)
assert somatic_calibration["valid"]

## 6. Piano diagnostico

Il default contiene 36 traiettorie brevi: 12 protocolli, ciascuno con seed distinti in train, validation e test. Nessuna finestra della stessa traiettoria viene distribuita tra split diversi. Le soglie evento sono ipotesi diagnostiche e i grafici finali restano obbligatori per la revisione.

In [ ]:
protocols = session.build_default_protocols()
protocol_table = pd.DataFrame([{
    "trajectory_id": item.trajectory_id,
    "category": item.category,
    "protocol": item.protocol,
    "seed": item.seed,
    "split": item.split,
    "duration_ms": item.duration_ms,
    "input_steps": len(item.actions_by_step),
} for item in protocols])
display(protocol_table)
display(protocol_table.groupby(["category", "split"]).size().unstack(fill_value=0))

## 7. Generazione delle transizioni

Per ogni boundary salviamo stato ricco, RNG, snapshot NEURON nativo e input ordinato. A 0,025 ms salviamo le sonde, variabili selezionate e — solo perché il dataset è piccolo — il voltaggio di tutti i 642 segmenti.

In [ ]:
dataset_manifest = session.generate_dataset(protocols)
display(dataset_manifest)
print("Transition store:", session.transition_path)

## 8. Validazione e branching

La validazione controlla NaN/Inf, forme, mapping, timestamp, split, hash, corrente somatica realmente erogata, copertura obbligatoria degli spike soma/AIS, coerenza microtraccia/boundary, replay di transizioni casuali, replay di un'intera traiettoria test con eventi e branching controllato.

In [ ]:
validation_report = session.validate_dataset(replay_count=3)
display(validation_report)
assert validation_report["valid"]
assert validation_report["test_trajectory_replay"]["events_match"]
assert validation_report["protocol_coverage"]["valid_current_delivery"]
assert validation_report["protocol_coverage"]["somatic_event_coverage_valid"]

## 9. Revisione visuale e artefatti

Aprire i quattro grafici di traiettoria nella cartella `figures/`. Le linee verticali indicano onset rilevati: una label non diventa definitiva finché queste tracce non sono state controllate visivamente.

In [ ]:
from IPython.display import Image, display

for figure_path in sorted(session.figures_dir.glob("*.png")):
    print(figure_path.name)
    display(Image(filename=str(figure_path)))

In [ ]:
required = [
    "burnin_report.json", "dataset_manifest.json",
    "somatic_current_smoke_test.json", "somatic_current_calibration.json",
    "event_definition_config.json", "manifest.json",
    "state_schema.json", "segments.parquet", "synapses.parquet",
    "transition_dataset.h5", "validation_report.json",
    "artifact_index.json",
]
artifact_table = pd.DataFrame([{
    "path": name,
    "exists": (OUTPUT_DIR / name).is_file(),
    "size_bytes": (OUTPUT_DIR / name).stat().st_size if (OUTPUT_DIR / name).is_file() else None,
} for name in required])
display(artifact_table)
assert artifact_table["exists"].all()

## 10. Archivio da scaricare

Su Kaggle l'archivio compare nel pannello **Files** sotto `/kaggle/working`. Se il pannello non si aggiorna, usare Refresh e poi il menu a tre puntini del file ZIP.

In [ ]:
archive_base = NOTEBOOK_ROOT / "hayflow_transition_dataset_diagnostic"
archive_path = session.package_artifacts(archive_base)
print("ZIP pronto:", archive_path)
print("Dimensione (MiB):", round(archive_path.stat().st_size / 1024**2, 2))
print("Kaggle: Files > Refresh > hayflow_transition_dataset_diagnostic.zip > Download")